# Universidad del Valle de Guatemala

## Departamento de Ciencias de la Computacion

## Inteligencia Artificial - Seccion 10



# Laboratorio No. 6



### Nadissa Vela - 23764

### Cristian Tunchez - 231359



## Task 2 - Implementacion Connect Four (4 en linea)



### Introduccion

En este laboratorio implementamos una IA para **Connect Four** utilizando **Minimax** con profundidad limitada.



Connect Four es un juego de suma cero en un tablero de **7 columnas x 6 filas**, donde gana quien conecta 4 fichas consecutivas (horizontal, vertical o diagonal).



La utilidad en estados terminales se define como:

- **+1000** si gana la IA.

- **-1000** si gana el oponente.

- **0** si hay empate.



En **Task 2.1** se implementa:

- La clase `Connect4` para modelar estado, acciones y terminalidad.

- El agente **Minimax recursivo** mediante `get_best_move(board, depth)`.

- Restriccion de profundidad recomendada: **d = 3** o **d = 4**.


## 1. Importaciones y constantes del entorno



En esta seccion se importan utilidades basicas y se definen constantes globales:

- Dimensiones del tablero (`ROWS`, `COLS`).

- Representacion de casillas vacias (`EMPTY`).

- Identificadores de jugadores (`PLAYER_1`, `PLAYER_2`).



Estas constantes se reutilizan en toda la implementacion para mantener consistencia del modelo.

In [46]:
from copy import deepcopy

import random



ROWS = 6

COLS = 7

EMPTY = " "

PLAYER_1 = "X"

PLAYER_2 = "O"

## 2. Clase Connect4: modelado del juego



Esta celda implementa la logica principal del entorno de juego:

- `actions(s)`: columnas validas para jugar.

- `make_move(col, player)`: aplica una jugada respetando gravedad.

- `succ(s, a)`: genera estado sucesor.

- `check_winner(player)`: detecta 4 en linea horizontal, vertical y diagonal.

- `is_terminal(s)`: determina fin del juego por victoria o empate.



Con esto se define formalmente el problema como juego de dos jugadores y suma cero.

In [47]:
class Connect4:

    def __init__(self, board=None):

        if board is None:

            self.board = [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]

        else:

            self.board = deepcopy(board)



    def copy(self):

        return Connect4(self.board)



    def actions(self):

        # Columnas validas: la celda superior aun esta vacia.

        return [c for c in range(COLS) if self.board[0][c] == EMPTY]



    def make_move(self, col, player):

        if col not in self.actions():

            return False

        for r in range(ROWS - 1, -1, -1):

            if self.board[r][col] == EMPTY:

                self.board[r][col] = player

                return True

        return False



    def succ(self, action, player):

        next_state = self.copy()

        next_state.make_move(action, player)

        return next_state



    def check_winner(self, player):

        b = self.board



        # Horizontal

        for r in range(ROWS):

            for c in range(COLS - 3):

                if b[r][c] == player and b[r][c + 1] == player and b[r][c + 2] == player and b[r][c + 3] == player:

                    return True



        # Vertical

        for r in range(ROWS - 3):

            for c in range(COLS):

                if b[r][c] == player and b[r + 1][c] == player and b[r + 2][c] == player and b[r + 3][c] == player:

                    return True



        # Diagonal descendente (\\)

        for r in range(ROWS - 3):

            for c in range(COLS - 3):

                if b[r][c] == player and b[r + 1][c + 1] == player and b[r + 2][c + 2] == player and b[r + 3][c + 3] == player:

                    return True



        # Diagonal ascendente (/)

        for r in range(3, ROWS):

            for c in range(COLS - 3):

                if b[r][c] == player and b[r - 1][c + 1] == player and b[r - 2][c + 2] == player and b[r - 3][c + 3] == player:

                    return True



        return False



    def is_terminal(self):

        return self.check_winner(PLAYER_1) or self.check_winner(PLAYER_2) or len(self.actions()) == 0



    def winner(self):

        if self.check_winner(PLAYER_1):

            return PLAYER_1

        if self.check_winner(PLAYER_2):

            return PLAYER_2

        return None



    def print_board(self):

        # Muestra fila superior primero con marco decorativo.

        line = "*" * (COLS * 4 + 5)

        print()

        print(line)

        for r in range(ROWS):

            print("* | " + " | ".join(self.board[r]) + " | *")

        print(line)

        print()

## 3. Funcion de evaluacion heuristica



Como Minimax se limita a profundidad `d`, no siempre llega a un estado terminal.

Por eso se implementa `evaluate_board(state, ai_player)`, que aproxima que tan bueno es un estado intermedio.



Criterios principales:

- Priorizar la columna central.

- Recompensar ventanas de 4 con 2 o 3 fichas propias.

- Penalizar amenazas del oponente (3 en linea con un espacio).

In [48]:
def evaluate_window(window, ai_player):
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    if window.count(ai_player) == 4:
        return 100
    if window.count(ai_player) == 3 and window.count(EMPTY) == 1:
        return 5
    if window.count(ai_player) == 2 and window.count(EMPTY) == 2:
        return 2

    if window.count(opponent) == 3 and window.count(EMPTY) == 1:
        return -4

    return 0


def evaluate_board(state, ai_player):
    # Evalua estado no terminal cuando depth == 0.
    score = 0
    b = state.board

    # Preferencia por columna central.
    center_col = COLS // 2
    center_values = [b[r][center_col] for r in range(ROWS)]
    score += center_values.count(ai_player) * 3

    # Horizontal
    for r in range(ROWS):
        row_vals = b[r]
        for c in range(COLS - 3):
            score += evaluate_window(row_vals[c:c+4], ai_player)

    # Vertical
    for c in range(COLS):
        col_vals = [b[r][c] for r in range(ROWS)]
        for r in range(ROWS - 3):
            score += evaluate_window(col_vals[r:r+4], ai_player)

    # Diagonal descendente (\)
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            score += evaluate_window([b[r+i][c+i] for i in range(4)], ai_player)

    # Diagonal ascendente (/ )
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            score += evaluate_window([b[r-i][c+i] for i in range(4)], ai_player)

    return score

## 4. Agente Minimax y seleccion de jugada



Aqui se implementa el algoritmo **Minimax recursivo**:

- Nodo MAX: turno de la IA (maximiza valor).

- Nodo MIN: turno del oponente (minimiza valor).

- Casos base: estado terminal o profundidad 0.



La funcion clave solicitada en el task es:

- `get_best_move(board, depth, ai_player)`



Para este laboratorio se recomienda `depth=3` o `depth=4` para controlar el tiempo de ejecucion sin poda alfa-beta.

In [49]:
def minimax(state, depth, maximizing_player, ai_player):
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    if state.is_terminal():
        winner = state.winner()
        if winner == ai_player:
            return None, 1000
        if winner == opponent:
            return None, -1000
        return None, 0

    if depth == 0:
        return None, evaluate_board(state, ai_player)

    valid_moves = state.actions()

    if maximizing_player:
        best_score = float('-inf')
        best_move = random.choice(valid_moves)

        for col in valid_moves:
            child = state.succ(col, ai_player)
            _, score = minimax(child, depth - 1, False, ai_player)
            if score > best_score:
                best_score = score
                best_move = col

        return best_move, best_score

    best_score = float('inf')
    best_move = random.choice(valid_moves)

    for col in valid_moves:
        child = state.succ(col, opponent)
        _, score = minimax(child, depth - 1, True, ai_player)
        if score < best_score:
            best_score = score
            best_move = col

    return best_move, best_score


def get_best_move(board, depth=4, ai_player=PLAYER_1):
    """
    board: matriz 6x7 (listas de listas) o instancia Connect4
    depth: recomendado 3 o 4 para Task 2.1
    """
    state = board if isinstance(board, Connect4) else Connect4(board)
    move, _ = minimax(state, depth, True, ai_player)
    return move

## 5. Simulacion de validacion



Esta seccion ejecuta una prueba rapida de funcionamiento:

- IA con Minimax contra un oponente aleatorio.

- Se imprime el tablero final y el ganador.



Objetivo: verificar que la logica del juego y la eleccion de jugadas del agente funcionan correctamente.

In [50]:
# Prueba rapida: IA vs jugador aleatorio
def play_ai_vs_random(depth=4, ai_player=PLAYER_1, seed=7):
    random.seed(seed)
    game = Connect4()
    current = PLAYER_1

    while not game.is_terminal():
        if current == ai_player:
            col = get_best_move(game, depth=depth, ai_player=ai_player)
        else:
            col = random.choice(game.actions())

        game.make_move(col, current)
        current = PLAYER_1 if current == PLAYER_2 else PLAYER_2

    game.print_board()
    w = game.winner()
    if w is None:
        return "Empate"
    return f"Gana jugador {w}"


resultado = play_ai_vs_random(depth=4, ai_player=PLAYER_1, seed=11)
print("Resultado:", resultado)


*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   | X |   |   |   | *
* |   |   |   | X |   |   |   | *
* |   | O |   | X |   |   |   | *
* | O | O |   | X | X | O |   | *
*********************************

Resultado: Gana jugador X


## 6. Task 2.2 - Optimizacion con Poda Alfa-Beta



En esta parte optimizamos el agente Minimax incorporando **Poda Alfa-Beta**.



Idea clave:

- `alpha`: mejor valor (cota inferior) que MAX puede garantizar hasta el momento.

- `beta`: mejor valor (cota superior) que MIN puede garantizar hasta el momento.

- Si en algun punto `alpha >= beta`, se **poda** ese subarbol porque ya no puede cambiar la decision final.



Ademas, para demostrar el impacto, se cuentan los nodos visitados por:

1. Minimax puro

2. Minimax con Alfa-Beta



La comparacion se hace sobre **el mismo estado del tablero** y la misma profundidad de busqueda.

In [51]:
# Implementacion instrumentada para comparar nodos visitados.

def minimax_count_nodes(state, depth, maximizing_player, ai_player, counter):

    counter["nodes"] += 1

    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2



    if state.is_terminal():

        winner = state.winner()

        if winner == ai_player:

            return None, 1000

        if winner == opponent:

            return None, -1000

        return None, 0



    if depth == 0:

        return None, evaluate_board(state, ai_player)



    valid_moves = state.actions()



    if maximizing_player:

        best_score = float("-inf")

        best_move = valid_moves[0]

        for col in valid_moves:

            child = state.succ(col, ai_player)

            _, score = minimax_count_nodes(child, depth - 1, False, ai_player, counter)

            if score > best_score:

                best_score = score

                best_move = col

        return best_move, best_score



    best_score = float("inf")

    best_move = valid_moves[0]

    for col in valid_moves:

        child = state.succ(col, opponent)

        _, score = minimax_count_nodes(child, depth - 1, True, ai_player, counter)

        if score < best_score:

            best_score = score

            best_move = col

    return best_move, best_score





def alphabeta_count_nodes(state, depth, alpha, beta, maximizing_player, ai_player, counter):

    counter["nodes"] += 1

    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2



    if state.is_terminal():

        winner = state.winner()

        if winner == ai_player:

            return None, 1000

        if winner == opponent:

            return None, -1000

        return None, 0



    if depth == 0:

        return None, evaluate_board(state, ai_player)



    valid_moves = state.actions()



    if maximizing_player:

        best_score = float("-inf")

        best_move = valid_moves[0]

        for col in valid_moves:

            child = state.succ(col, ai_player)

            _, score = alphabeta_count_nodes(child, depth - 1, alpha, beta, False, ai_player, counter)

            if score > best_score:

                best_score = score

                best_move = col



            # Actualiza alpha en nodo MAX.

            alpha = max(alpha, best_score)



            # Poda: MIN ya tiene una opcion mejor o igual en ancestros.

            if alpha >= beta:

                break



        return best_move, best_score



    best_score = float("inf")

    best_move = valid_moves[0]

    for col in valid_moves:

        child = state.succ(col, opponent)

        _, score = alphabeta_count_nodes(child, depth - 1, alpha, beta, True, ai_player, counter)

        if score < best_score:

            best_score = score

            best_move = col



        # Actualiza beta en nodo MIN.

        beta = min(beta, best_score)



        # Poda: MAX ya tiene una opcion mejor o igual en ancestros.

        if alpha >= beta:

            break



    return best_move, best_score





def get_best_move_alphabeta(board, depth=4, ai_player=PLAYER_1):

    state = board if isinstance(board, Connect4) else Connect4(board)

    move, _ = alphabeta_count_nodes(

        state=state,

        depth=depth,

        alpha=float("-inf"),

        beta=float("inf"),

        maximizing_player=True,

        ai_player=ai_player,

        counter={"nodes": 0},

    )

    return move

## 7. Demostracion: Minimax vs Alfa-Beta (nodos visitados)



Se construye un estado intermedio fijo del tablero y se ejecutan ambos algoritmos con la misma profundidad.



Salida esperada:

- Mejor movimiento sugerido por cada algoritmo.

- Puntaje estimado por cada algoritmo.

- Nodos visitados por Minimax puro.

- Nodos visitados por Alfa-Beta.

- Porcentaje de reduccion de nodos (deberia ser significativo).

In [52]:
def build_demo_state():

    # Secuencia fija de jugadas para crear un estado no terminal reproducible.

    # PLAYER_1 y PLAYER_2 alternan turnos.

    moves = [3, 2, 3, 2, 4, 2, 4, 1, 5, 1]

    game = Connect4()

    current = PLAYER_1



    for col in moves:

        game.make_move(col, current)

        current = PLAYER_1 if current == PLAYER_2 else PLAYER_2



    return game





depth_test = 4

ai_player = PLAYER_1

state_demo = build_demo_state()



print("Tablero de prueba (estado compartido):")

state_demo.print_board()



# Minimax puro con contador.

counter_minimax = {"nodes": 0}

move_mm, score_mm = minimax_count_nodes(

    state=state_demo,

    depth=depth_test,

    maximizing_player=True,

    ai_player=ai_player,

    counter=counter_minimax,

)



# Alfa-Beta con contador.

counter_ab = {"nodes": 0}

move_ab, score_ab = alphabeta_count_nodes(

    state=state_demo,

    depth=depth_test,

    alpha=float("-inf"),

    beta=float("inf"),

    maximizing_player=True,

    ai_player=ai_player,

    counter=counter_ab,

)



nodes_mm = counter_minimax["nodes"]

nodes_ab = counter_ab["nodes"]

reduction = 0.0 if nodes_mm == 0 else (1 - (nodes_ab / nodes_mm)) * 100



print(f"Minimax puro -> mejor mov: {move_mm}, score: {score_mm}, nodos: {nodes_mm}")

print(f"Alfa-Beta    -> mejor mov: {move_ab}, score: {score_ab}, nodos: {nodes_ab}")

print(f"Reduccion de nodos: {reduction:.2f}%")



if move_mm == move_ab:

    print("Ambos algoritmos coinciden en la decision (esperado).")

else:

    print("Nota: hubo diferencia en la decision; revise orden de exploracion y heuristica.")

Tablero de prueba (estado compartido):

*********************************
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   |   |   |   |   |   | *
* |   |   | O |   |   |   |   | *
* |   | O | O | X | X |   |   | *
* |   | O | O | X | X | X |   | *
*********************************

Minimax puro -> mejor mov: 2, score: 1000, nodos: 1743
Alfa-Beta    -> mejor mov: 2, score: 1000, nodos: 377
Reduccion de nodos: 78.37%
Ambos algoritmos coinciden en la decision (esperado).


Con la Poda Alfa-Beta:

- Se mantiene la logica de decision de Minimax.

- Se evita explorar ramas que no pueden mejorar la decision.

- Se reduce de forma importante la cantidad de nodos visitados.



Esto permite buscar mas profundo en el mismo tiempo de computo, mejorando la competitividad del agente.